In [9]:
# Suppress verbose logging from earthaccess and fsspec
import logging
logging.getLogger('earthaccess').setLevel(logging.WARNING)
logging.getLogger('fsspec').setLevel(logging.WARNING)

## Define functions

In [2]:
"""Helper functions for PACE Hackweek Validation Tutorial.

Authors:
    James Allen and Anna Windle
"""

import datetime
import os
import re
from pathlib import Path

import earthaccess
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import matplotlib.style as style
import h5py
import numpy as np
import pandas as pd
import seaborn as sns
import xarray as xr
from matplotlib.ticker import FuncFormatter
from scipy import odr, stats
import requests

# AERONET-OC Download Constants
# Valid AERONET-OC site list
DF_AERONET_SITES = pd.read_csv(
    "https://aeronet.gsfc.nasa.gov/aeronet_locations_v3.txt",
    delimiter=",",
    skiprows=1
    )
AERONET_SITES = list(DF_AERONET_SITES["Site_Name"].sort_values())
OCEAN_SITES = [
    "AAOT",
    "Abu_Al_Bukhoosh",
    "ARIAKE_TOWER",
    "Bahia_Blanca",
    "Banana_River",
    "Blyth_NOAH",
    "Casablanca_Platform",
    "Chesapeake_Bay",
    "COVE_SEAPRISM",
    "Galata_Platform",
    "Gloria",
    "GOT_Seaprism",
    "Grizzly_Bay",
    "Gustav_Dalen_Tower",
    "Helsinki_Lighthouse",
    "Ieodo_Station",
    "Irbe_Lighthouse",
    "Kemigawa_Offshore",
    "Lake_Erie",
    "Lake_Okeechobee",
    "Lake_Okeechobee_N",
    "LISCO",
    "Lucinda",
    "MVCO",
    "Palgrunden",
    "PLOCAN_Tower",
    "RdP-EsNM",
    "Sacramento_River",
    "San_Marco_Platform",
    "Section-7_Platform",
    "Socheongcho",
    "South_Greenbay",
    "Thornton_C-power",
    "USC_SEAPRISM",
    "Venise",
    "WaveCIS_Site_CSI_6",
    "Zeebrugge-MOW1",
]

# Get subset of AERONET columns to make it a bit more manageable (also rename)
AOC_KEEP_COLS = [
    "AERONET_Site",
    "field_datetime",
    "Site_Latitude(Degrees)",
    "Site_Longitude(Degrees)",
    "Solar_Zenith_Angle[400nm]",
]
COLUMN_RENAME = {
    "Site_Latitude(Degrees)": "field_latitude",
    "Site_Longitude(Degrees)": "field_longitude",
    "AERONET_Site": "field_site",
    "Solar_Zenith_Angle[400nm]": "field_solar_zenith",
}

# Bland-Altman/Scatterplot Constants
# Plot colors, font sizes
COLOR_PALETTE = sns.color_palette("colorblind")
COLOR_SCATTER = COLOR_PALETTE[0]
COLOR_LINE = "black"  # Was "black"
COLOR_LOA = COLOR_PALETTE[2]  # Was "green"
COLOR_FITLINE = COLOR_PALETTE[1]  # Was "magenta"
SIZE_TITLE = 24
SIZE_AXLABEL = 20
SIZE_TEXTLABEL = 14
SHOW_LEGEND = False

# Update some defaults
plt.rcParams.update({"figure.dpi": 300})
sns.set_style("ticks", rc={"figure.dpi": 300})
sns.set_context("notebook", font_scale=1.45)

# Satellite Matchup Constants
# Short names for earthaccess lookup
SAT_LOOKUP = {
    "PACE_AOP": "PACE_OCI_L2_AOP",
    "PACE_IOP": "PACE_OCI_L2_IOP",
    "PACE_BGC": "PACE_OCI_L2_BGC",
    "PACE_PAR": "PACE_OCI_L2_PAR",
    "AQUA": "MODISA_L2_OC",
    "TERRA": "MODIST_L2_OC",
    "NOAA-20": "VIIRSJ1_L2_OC",
    "NOAA-21": "VIIRSJ2_L2_OC",
    "SUOMI-NPP": "VIIRSN_L2_OC",
}

# List l2 flags, then build them into a dict
l2_flags_list = [
    "ATMFAIL",
    "LAND",
    "PRODWARN",
    "HIGLINT",
    "HILT",
    "HISATZEN",
    "COASTZ",
    "SPARE",
    "STRAYLIGHT",
    "CLDICE",
    "COCCOLITH",
    "TURBIDW",
    "HISOLZEN",
    "SPARE",
    "LOWLW",
    "CHLFAIL",
    "NAVWARN",
    "ABSAER",
    "SPARE",
    "MAXAERITER",
    "MODGLINT",
    "CHLWARN",
    "ATMWARN",
    "SPARE",
    "SEAICE",
    "NAVFAIL",
    "FILTER",
    "SPARE",
    "BOWTIEDEL",
    "HIPOL",
    "PRODFAIL",
    "SPARE",
]
L2_FLAGS = {flag: 1 << idx for idx, flag in enumerate(l2_flags_list)}

# Bailey and Werdell 2006 exclusion criteria
EXCLUSION_FLAGS = [
    "LAND",
    "HIGLINT",
    "HILT",
    "STRAYLIGHT",
    "CLDICE",
    "ATMFAIL",
    "LOWLW",
    "FILTER",
    "NAVFAIL",
    "NAVWARN",
]

# OCSSW Dataroot folder for tables
# OCDATAROOT = Path(os.environ.get("OCSSWROOT")).resolve() / "share"
OCDATAROOT = "/private/tmp/ocssw/share/"
OCI_SENSOR_FILE = OCDATAROOT + "oci/msl12_sensor_info.dat"

##---------------------------------------------------------------------------##
#                              General Utilities                              #
##---------------------------------------------------------------------------##


def get_f0(wavelengths=None, window_size=10):
    """Load the OCI sensor file and return F0.

    Defaults to returning the full table. Input obs_time to correct for the
    Earth-Sun distance.

    Parameters
    ----------
    sensor_file : str or pathlib.Path
        Path to the OCI satellite sensor file containing wavelengths and F0.
    wavelengths : array-like, optional
        Wavelengths at which to compute the average irradiance.
        If None, returns the full wavelength and irradiance table.
    window_size : int, optional
        Bandpass filter size for mean filtering to selected wavelengths, in nm.

    Returns
    -------
    tuple of np.ndarray
        A tuple containing:
        - f0_spectra : np.ndarray
            The extraterrestrial solar irradiance, in uW/cm^2/nm.
        - f0_wave : np.ndarray
            The corresponding wavelengths, in nm.

    """
    with open(OCI_SENSOR_FILE, "r") as file_in:
        for line in file_in:
            if "Nbands" in line:
                (key, nbands) = line.split("=")
                break

    wl = np.zeros(int(nbands), dtype=float)
    f0 = np.zeros(int(nbands), dtype=float)
    with open(OCI_SENSOR_FILE, "r") as file_in:
        for line in file_in:
            if "=" in line:
                (key, value) = line.split("=")
                if "Lambda" in key:
                    idx = re.findall(r"\d+", key)
                    wvlidx = int(idx[0]) - 1
                    wl[wvlidx] = float(value)
                if "F0" in key:
                    idx = re.findall(r"\d+", key)
                    wvlidx = int(idx[1]) - 1
                    f0[wvlidx] = float(value)

    if wavelengths is not None:
        f0_wave = np.array(wavelengths)
        f0_spectra = bandpass_avg(f0, wl, window_size, f0_wave)
    else:
        f0_wave = wl
        f0_spectra = f0

    return f0_spectra, f0_wave


def bandpass_avg(
        data,
        input_wavelengths,
        window_size=10,
        target_wavelengths=None
        ):
    """Apply a band-pass filter to the data.

    Parameters
    ----------
    data : np.ndarray
        1D or 2D array containing the spectral data (samples x wavelengths).
        If 1D, it's assumed to be a single sample.
    input_wavelengths : np.ndarray
        1D array of wavelength values corresponding to the columns of data.
    window_size : int, optional
        Size of the window to use for averaging. Default is 10 nm.
    target_wavelengths : np.ndarray, optional
        1D array of target wavelengths for filtered values.
        If None, the input wavelengths are used.

    Returns
    -------
    np.ndarray
        1D or 2D array containing the band-pass 
        data.

    """
    data = np.atleast_2d(data)
    half_window = window_size / 2
    num_samples, num_input_wavelengths = data.shape
    if target_wavelengths is None:
        target_wavelengths = input_wavelengths

    filtered_data = np.empty((num_samples, len(target_wavelengths))) * np.nan

    for idx, target_wl in enumerate(target_wavelengths):
        start = target_wl - half_window
        end = target_wl + half_window
        cols_in_range = np.where(
            (input_wavelengths >= start) & (input_wavelengths <= end)
        )[0]
        if cols_in_range.size > 0:
            filtered_data[:, idx] = np.nanmean(data[:, cols_in_range], axis=1)

    return filtered_data if num_samples > 1 else filtered_data.flatten()


def get_column_prods(df, type_prefix):
    """Process a dataframe to create a dictionary of data products.

    Parameters
    ----------
    df : pandas DataFrame
        Extracted dataframes from read_extract_file
    type_prefix : str
        Prefix to identify the product columns, e.g. "aoc"

    Returns
    -------
    data_dict
        dictionary mapping data product with their wavelengths and columns.

    """
    data_dict = {}
    pattern = rf"{type_prefix}_(\w+?)(\d*\.?\d+)?$"

    for col in df.columns:
        match = re.match(pattern, col)
        if match:
            product = match.group(1)
            wavelength = match.group(2) if match.group(2) else None
            if product not in data_dict:
                data_dict[product] = {"wavelengths": [], "columns": []}
            data_dict[product]["columns"].append(col)
            if wavelength:
                if "." in wavelength:
                    data_dict[product]["wavelengths"].append(float(wavelength))
                else:
                    data_dict[product]["wavelengths"].append(int(wavelength))
    return data_dict


def read_sb(filename_sb):
    """Read SeaBASS file and returns just the data.

    Input
    -----
    filename_sb : str
        path to seabass file

    Output
    ------
    data : pandas dataframe object
        seabass data from file
    """
    with open(filename_sb, "r") as file:
        lines = [line.rstrip() for line in file]

    # Parse headers, get index where they end
    idx_endheader = [index for index, value in enumerate(lines)
                     if value == "/end_header"]
    header_lines = lines[1:idx_endheader[0]]
    headers = dict()
    comments = []
    for header_line in header_lines:
        if header_line.startswith("!"):
            # Separate out the comments
            comments.append(header_line)
        else:
            # Split the header and add to the dictionary
            key, value = header_line.split("=", 1)
            headers[key[1:]] = value  # Remove leading "/" from key

    # Pull data into pandas dataframe
    data = pd.read_csv(filename_sb,
                       skiprows=idx_endheader[0]+1,
                       names=headers["fields"].split(","),
                       na_values=headers["missing"])

    # Index by datetime
    get_sb_datetime(data)

    return data


def get_sb_datetime(df):
    """Parse datetime from different combinations of dates and times."""
    if all(col in df.columns for col in ["year", "month", "day",
                                         "hour", "minute", "second"]):
        df["datetime"] = pd.to_datetime(df[["year", "month", "day",
                                            "hour", "minute", "second"]])
    elif all(col in df.columns for col in ["year", "month", "day", "time"]):
        df["datetime"] = pd.to_datetime(
            df["year"].astype(str) + df["month"].astype(str).str.zfill(2)
            + df["day"].astype(str).str.zfill(2) + ' ' + df["time"])
    elif all(col in df.columns for col in ["date", "time"]):
        df["datetime"] = pd.to_datetime(
            df["date"].astype(str) + ' ' + df["time"])
    elif all(col in df.columns for col in ["year", "month", "day"]):
        df["datetime"] = pd.to_datetime(df[["year", "month", "day"]])
    elif all(col in df.columns for col in ["date", "hour",
                                           "minute", "second"]):
        df["datetime"] = pd.to_datetime(
            df["date"].astype(str) + ' ' + df["hour"].astype(str).str.zfill(2)
            + ':' + df["minute"].astype(str).str.zfill(2) + ':'
            + df["second"].astype(str).str.zfill(2))
    else:
        print("Unrecognized date/time format in DataFrame columns."
              "\nMay be a profile, but doublecheck.")
        return

    # Reindex the dataframe with the new datetime
    df.set_index("datetime", inplace=True)


##---------------------------------------------------------------------------##
#                             Satellite Utilities                             #
##---------------------------------------------------------------------------##


def parse_quality_flags(flag_value):
    """Parse bitwise flag into a list of flag names.

    Parameters
    ----------
    flag_value : int
        The integer representing the combined bitwise quality flags.

    Returns
    -------
    list of str
        List of flag names that are set in the flag_value.

    """
    return [
        flag_name for flag_name, value in L2_FLAGS.items()
        if (flag_value & value) != 0
    ]


def get_fivebyfive_Rrs(file, latitude, longitude, par, wavelengths, rrs_wavelengths):
    """Get stats on 5x5 box around station coordinates of a satellite granule.

    This checks l2flags and runs statistics on valid pixels and returns their
    valid count, the coefficient of variance (cv), and the Rrs values.

    Parameters
    ----------
    file : earthaccess granule object
        Satellite granule from earthaccess.
    latitude : float
        In decimal degrees for Aeronet-OC site for matchups
    longitude : float
        In decimal degrees (negative West) for Aeronet-OC site for matchups
    wavelengths ; numpy array
        Desired Rrs wavelengths to match
    rrs_wavelengths ; numpy array
        Rrs wavelengths (from wavelength_3d for OCI)

    Returns
    -------
    dict
        A dictionary of the processed 5x5 box with:
            - "sat_datetime": pd.datetime
                Datetime of the overall granule start time
            - "sat_cv": float
                Median coefficient of variation of Rrs(405nm - 570nm)
            - "sat_latitude": float
                Latitude of center pixel
            - "sat_longitude": float
                Longitude of center pixel
            - "sat_pixel_valid": float
                Number of valid pixels in 5x5 box based on l2 flags

    Notes
    -----
    This is set to use just Rrs data for the demo. As an exercise, make this
    function more generalized by adding an input for the desired product and
    removing the wavelength dependency (if not needed) as well as the cv
    calculation. This will also require refactoring the `match_data` function.
    """
    with xr.open_dataset(file, group="navigation_data") as ds_nav:
        sat_lat = ds_nav["latitude"].values
        sat_lon = ds_nav["longitude"].values

    # Calculate the Euclidean distance for 2D lat/lon arrays
    distances = np.sqrt((sat_lat - latitude) ** 2 + (sat_lon - longitude) ** 2)

    # Find the index of the minimum distance
    # Dimensions are (lines, pixels)
    min_dist_idx = np.unravel_index(np.argmin(distances), distances.shape)
    center_line, center_pixel = min_dist_idx

    # Get indices for a 5x5 box around the center pixel
    line_start = max(center_line - 2, 0)
    line_end = min(center_line + 2 + 1, sat_lat.shape[0])
    pixel_start = max(center_pixel - 2, 0)
    pixel_end = min(center_pixel + 2 + 1, sat_lat.shape[1])

    # Extract the data
    # NOTE: This is hard-coded to Rrs from an L2 AOP file.
    with xr.open_dataset(file, group="geophysical_data") as ds_data:
        rrs_data = (
            ds_data[par].isel(
                number_of_lines=slice(line_start, line_end),
                pixels_per_line=slice(pixel_start, pixel_end),
            ).values
        )
        flags_data = (
            ds_data["l2_flags"].isel(
                number_of_lines=slice(line_start, line_end),
                pixels_per_line=slice(pixel_start, pixel_end),
            ).values
        )
        
        # Select only the desired wavelengths
        rrs_indices = [np.where(rrs_wavelengths == wl)[0][0] for wl in wavelengths]
        rrs_data = rrs_data[:, :, rrs_indices]

    # Calculate the bitwise OR of all flags in EXCLUSION_FLAGS to get a mask
    exclude_mask = sum(L2_FLAGS[flag] for flag in EXCLUSION_FLAGS)

    # Create a boolean mask
    # True means the flag value does not contain any of the EXCLUSION_FLAGS
    valid_mask = np.bitwise_and(flags_data, exclude_mask) == 0

    # Get stats and averages
    if valid_mask.any() and np.isnan(rrs_data[valid_mask]).all() == False:
        rrs_valid = rrs_data[valid_mask]
        rrs_std_initial = np.std(rrs_valid, axis=0)
        rrs_mean_initial = np.mean(rrs_valid, axis=0)

        # Exclude spectra > 1.5 stdevs away
        std_mask = np.all(
            np.abs(rrs_valid - rrs_mean_initial) <= 1.5 * rrs_std_initial,
            axis=1
        )
        rrs_std = np.std(rrs_valid[std_mask], axis=0)
        rrs_mean = np.mean(rrs_valid[std_mask], axis=0).flatten()
    else:
        valid_mask[:] = False
        rrs_mean = np.nan * np.empty_like(wavelengths)

    # Put in dictionary of the row
    row = {
        "sat_datetime": pd.to_datetime(
            file.granule["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"],
            utc=0
        ),
        "sat_latitude": sat_lat[center_line, center_pixel],
        "sat_longitude": sat_lon[center_line, center_pixel],
        "sat_pixel_valid": np.sum(valid_mask),
    }

    # Add mean spectra to the row dictionary
    for wavelength, mean_value in zip(wavelengths, rrs_mean):
        key = f"sat_{par.lower()}{int(wavelength)}"
        row[key] = mean_value

    return row


def get_fivebyfive_PAR(file, latitude, longitude):
    """Get stats on 5x5 box around station coordinates of a satellite granule.

    This checks l2flags and runs statistics on valid pixels and returns their
    valid count, the coefficient of variance (cv), and the Rrs values.

    Parameters
    ----------
    file : earthaccess granule object
        Satellite granule from earthaccess.
    latitude : float
        In decimal degrees for Aeronet-OC site for matchups
    longitude : float
        In decimal degrees (negative West) for Aeronet-OC site for matchups

    Returns
    -------
    dict
        A dictionary of the processed 5x5 box with:
            - "sat_datetime": pd.datetime
                Datetime of the overall granule start time
            - "sat_cv": float
                Median coefficient of variation of Rrs(405nm - 570nm)
            - "sat_latitude": float
                Latitude of center pixel
            - "sat_longitude": float
                Longitude of center pixel
            - "sat_pixel_valid": float
                Number of valid pixels in 5x5 box based on l2 flags

    Notes
    -----
    This is set to use just Rrs data for the demo. As an exercise, make this
    function more generalized by adding an input for the desired product and
    removing the wavelength dependency (if not needed) as well as the cv
    calculation. This will also require refactoring the `match_data` function.
    """
    with xr.open_dataset(file, group="navigation_data") as ds_nav:
        sat_lat = ds_nav["latitude"].values
        sat_lon = ds_nav["longitude"].values

    # Calculate the Euclidean distance for 2D lat/lon arrays
    distances = np.sqrt((sat_lat - latitude) ** 2 + (sat_lon - longitude) ** 2)

    # Find the index of the minimum distance
    # Dimensions are (lines, pixels)
    min_dist_idx = np.unravel_index(np.argmin(distances), distances.shape)
    center_line, center_pixel = min_dist_idx

    # Get indices for a 5x5 box around the center pixel
    line_start = max(center_line - 2, 0)
    line_end = min(center_line + 2 + 1, sat_lat.shape[0])
    pixel_start = max(center_pixel - 2, 0)
    pixel_end = min(center_pixel + 2 + 1, sat_lat.shape[1])

    # Extract the data
    # NOTE: Using "par_day_planar_above" product for PAR data, which I assume is what standard OC PAR is...
    with xr.open_dataset(file, group="geophysical_data") as ds_data:
        par_data = (
            ds_data["par_day_planar_above"].isel(
                number_of_lines=slice(line_start, line_end),
                pixels_per_line=slice(pixel_start, pixel_end),
            ).values
        )
        flags_data = (
            ds_data["l2_flags"].isel(
                number_of_lines=slice(line_start, line_end),
                pixels_per_line=slice(pixel_start, pixel_end),
            ).values
        )

    # Calculate the bitwise OR of all flags in EXCLUSION_FLAGS to get a mask
    exclude_mask = sum(L2_FLAGS[flag] for flag in EXCLUSION_FLAGS)

    # Create a boolean mask
    # True means the flag value does not contain any of the EXCLUSION_FLAGS
    valid_mask = np.bitwise_and(flags_data, exclude_mask) == 0

    # Get stats and averages
    if valid_mask.any():
        par_valid = par_data[valid_mask]
        par_mean = np.mean(par_valid, axis=0)
    else:
        par_mean = np.nan

    # Put in dictionary of the row
    row = {
        "sat_datetime": pd.to_datetime(
            file.granule["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"],
            utc=0
        ),
        "sat_latitude": sat_lat[center_line, center_pixel],
        "sat_longitude": sat_lon[center_line, center_pixel],
        "sat_pixel_valid": np.sum(valid_mask),
        "sat_par": par_mean
    }

    return row


def get_fivebyfive_SUOMI(file, latitude, longitude, wavelengths):
    """Get stats on 5x5 box around station coordinates of a satellite granule.

    This checks l2flags and runs statistics on valid pixels and returns their
    valid count, the coefficient of variance (cv), and the Rrs values.

    Parameters
    ----------
    file : earthaccess granule object
        Satellite granule from earthaccess.
    latitude : float
        In decimal degrees for Aeronet-OC site for matchups
    longitude : float
        In decimal degrees (negative West) for Aeronet-OC site for matchups

    Returns
    -------
    dict
        A dictionary of the processed 5x5 box with:
            - "sat_datetime": pd.datetime
                Datetime of the overall granule start time
            - "sat_cv": float
                Median coefficient of variation of Rrs(405nm - 570nm)
            - "sat_latitude": float
                Latitude of center pixel
            - "sat_longitude": float
                Longitude of center pixel
            - "sat_pixel_valid": float
                Number of valid pixels in 5x5 box based on l2 flags

    Notes
    -----
    This is set to use just Rrs data for the demo. As an exercise, make this
    function more generalized by adding an input for the desired product and
    removing the wavelength dependency (if not needed) as well as the cv
    calculation. This will also require refactoring the `match_data` function.
    """
    with xr.open_dataset(file, group="navigation_data") as ds_nav:
        sat_lat = ds_nav["latitude"].values
        sat_lon = ds_nav["longitude"].values

    # Calculate the Euclidean distance for 2D lat/lon arrays
    distances = np.sqrt((sat_lat - latitude) ** 2 + (sat_lon - longitude) ** 2)

    # Find the index of the minimum distance
    # Dimensions are (lines, pixels)
    min_dist_idx = np.unravel_index(np.argmin(distances), distances.shape)
    center_line, center_pixel = min_dist_idx

    # Get indices for a 5x5 box around the center pixel
    line_start = max(center_line - 2, 0)
    line_end = min(center_line + 2 + 1, sat_lat.shape[0])
    pixel_start = max(center_pixel - 2, 0)
    pixel_end = min(center_pixel + 2 + 1, sat_lat.shape[1])

    # Extract the data
    # NOTE: Using "par_day_planar_above" product for PAR data, which I assume is what standard OC PAR is...
    with xr.open_dataset(file, group="geophysical_data") as ds_data:
        # Extract Rrs data for each wavelength
        rrs_data_dict = {}
        for wl in wavelengths:
            rrs_key = f"Rrs_{int(wl)}"
            if rrs_key in ds_data:
                rrs_data_dict[wl] = ds_data[rrs_key].isel(
                    number_of_lines=slice(line_start, line_end),
                    pixels_per_line=slice(pixel_start, pixel_end),
                ).values
        par_data = (
            ds_data["par"].isel(
                number_of_lines=slice(line_start, line_end),
                pixels_per_line=slice(pixel_start, pixel_end),
            ).values
        )
        kd_data = (
            ds_data["Kd_490"].isel(
                number_of_lines=slice(line_start, line_end),
                pixels_per_line=slice(pixel_start, pixel_end),
            ).values
        )
        flags_data = (
            ds_data["l2_flags"].isel(
                number_of_lines=slice(line_start, line_end),
                pixels_per_line=slice(pixel_start, pixel_end),
            ).values
        )

    # Calculate the bitwise OR of all flags in EXCLUSION_FLAGS to get a mask
    exclude_mask = sum(L2_FLAGS[flag] for flag in EXCLUSION_FLAGS)

    # Create a boolean mask
    # True means the flag value does not contain any of the EXCLUSION_FLAGS
    valid_mask = np.bitwise_and(flags_data, exclude_mask) == 0

    # Get stats and averages
    if valid_mask.any():
        rrs_mean = np.full(len(wavelengths), np.nan)
        for i, wl in enumerate(wavelengths):
            rrs_valid = rrs_data_dict[wl][valid_mask]
            rrs_mean[i] = np.mean(rrs_valid)
        par_valid = par_data[valid_mask]
        par_mean = np.mean(par_valid, axis=0)
        kd_valid = kd_data[valid_mask]
        kd_mean = np.mean(kd_valid, axis=0)
    else:
        rrs_mean = np.full(len(wavelengths), np.nan)
        par_mean = np.nan
        kd_mean = np.nan
        

    # Put in dictionary of the row
    row = {
        "sat_datetime": pd.to_datetime(
            file.granule["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"],
            utc=0
        ),
        "sat_latitude": sat_lat[center_line, center_pixel],
        "sat_longitude": sat_lon[center_line, center_pixel],
        "sat_pixel_valid": np.sum(valid_mask),
        "sat_rrs_band1": rrs_mean[0],
        "sat_rrs_band2": rrs_mean[1],
        "sat_rrs_band3": rrs_mean[2],
        "sat_rrs_band4": rrs_mean[3],
        "sat_rrs_band5": rrs_mean[4],
        "sat_kd490": kd_mean,
        "sat_par": par_mean
    }

    return row

def get_sat_matchups(
    start_date,
    end_date,
    latitude,
    longitude,
    wavelengths="all",
    par="Rrs",
    sat="PACE_AOP",
    selected_dates=None
):
    """Make satellite timeseries of matchups from single station.

    Caution: If the date or coordinates aren't formatted correctly, it might
    pull a huge granule list and take forever to run. If it takes more than 45
    seconds to print the number of granules, just kill the process.

    Uses the earthaccess package. Defaults to the PACE OCI L2 IOP datasets,
    but other satellites can be used if they have a corresponding short_name
    in the SAT_LOOKUP dictionary.

    Workflow:
        1. Get list of matchup granules
        2. Loop through each file and:
            2a. Find closest pixel to station, extract 5x5 pixel box
            2b. Exclude pixels based on l2_flags
            2c. Filtered mean to get single spectra
            2d. Compute statistics and save data row
        3. Organize output pandas dataframe

    Parameters
    ----------
    start_date : datetime or str
        Beginning of Aeronet data to run.
    end_date : datetime or str, optional
        End of Aeronet data to run.
    latitude : float
        In decimal degrees for Aeronet-OC site for matchups
    longitude : float
        In decimal degrees (negative West) for Aeronet-OC site for matchups
    sat : str
        Name of satellite to search. Must be in SAT_LOOKUP dict constant.
    selected_dates : list of str, optional
        If given, only pull granules if the dates are in this list

    Returns
    -------
    pandas DataFrame object
        Flattened table of all satellite granule matchups.

    """
    # Look up short name from constants
    if sat not in SAT_LOOKUP.keys():
        raise ValueError(
            f"{sat} is not in the lookup dictionary. Available "
            f"sats are: {', '.join(SAT_LOOKUP)}"
        )
    short_name = SAT_LOOKUP[sat]

    # Format search parameters
    time_bounds = (f"{start_date}T00:00:00", f"{end_date}T23:59:59")

    # Run Earthaccess data search
    results = earthaccess.search_data(
        point=(longitude, latitude),
        temporal=time_bounds,
        short_name=short_name
    )
    if selected_dates is not None:
        filtered_results = [
            result
            for result in results
            if result["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"][:10]
            in selected_dates
        ]
        print(f"    Filtered to {len(filtered_results)} Granules.")
        files = earthaccess.open(filtered_results, pqdm_kwargs={"disable": True})
    else:
        files = earthaccess.open(results)

    # Get 5x5 pixel data
    if sat == "PACE_AOP" or sat == "PACE_IOP":
        if len(files) > 0:
            # Pull out all available Rrs wavelengths
            with xr.open_dataset(files[0], group="sensor_band_parameters") as ds_bands:
                rrs_wavelengths = ds_bands["wavelength_3d"].values
            
            # Select wavelengths or interest
            if wavelengths is None or wavelengths == "all":
                wavelengths = rrs_wavelengths
            else:
                # Get nearest wavelengths to desired input wavelengths
                nearest_wavelengths = []
                for target_wl in wavelengths:
                    nearest_wl = rrs_wavelengths[np.abs(rrs_wavelengths - target_wl).argmin()]
                    nearest_wavelengths.append(nearest_wl)
                wavelengths = np.array(nearest_wavelengths)
            
            # Loop through files and process
            sat_rows = []
            for idx, file in enumerate(files):
                granule_date = pd.to_datetime(
                    file.granule["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]
                )
                print(f"    Running Granule: {granule_date}")
                row = get_fivebyfive_Rrs(file, latitude, longitude, par, wavelengths, rrs_wavelengths)
                sat_rows.append(row)
        else:
            sat_rows = []
            
    elif sat == "PACE_PAR":
        # Loop through files and process
        sat_rows = []
        for idx, file in enumerate(files):
            granule_date = pd.to_datetime(
                file.granule["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]
            )
            print(f"    Running Granule: {granule_date}")
            row = get_fivebyfive_PAR(file, latitude, longitude)
            sat_rows.append(row)
    
    elif sat == "SUOMI-NPP" or sat == "NOAA-20" or sat == "NOAA-21" or sat == "AQUA":
        if len(files) > 0:
            # Pull out all available Rrs wavelengths
            with xr.open_dataset(files[0], group="sensor_band_parameters") as ds_bands:
                rrs_wavelengths = ds_bands["wavelength"].values
                
            # Select wavelengths or interest
            if wavelengths is None or wavelengths == "all":
                wavelengths = rrs_wavelengths
            else:
                # Get nearest wavelengths to desired input wavelengths
                nearest_wavelengths = []
                for target_wl in wavelengths:
                    nearest_wl = rrs_wavelengths[np.abs(rrs_wavelengths - target_wl).argmin()]
                    nearest_wavelengths.append(nearest_wl)
                wavelengths = np.array(nearest_wavelengths)
                
            # print(f"all available rrs_wavelengths: {rrs_wavelengths}")
            # print(f"nearest wavelengths found: {wavelengths}")
                
            # Loop through files and process
            sat_rows = []
            for idx, file in enumerate(files):
                granule_date = pd.to_datetime(
                    file.granule["umm"]["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]
                )
                print(f"    Running Granule: {granule_date}")
                row = get_fivebyfive_SUOMI(file, latitude, longitude, wavelengths)
                sat_rows.append(row)
        else:
            sat_rows = []
        
    return pd.DataFrame(sat_rows)
    

## ct182 tags > PACE matchups


### Creat profile list

In [1]:
fileDir = "/Volumes/Data/Work/SEALTAGS/Prydz/ct182/v5_20251106/PROCESSED/NC/"

df_profiles = pd.DataFrame(columns=["SEALTAG_NC_FILE", "TAG_ID", "PROFILE_NUM", "DATE", "LATITUDE", "LONGITUDE","ISDAYTIME","PROCESSED"])

# Loop through profile information excel files in directory
for file in sorted(os.listdir(fileDir)):
    if file.endswith(".xlsx") and not file.startswith("~$"):
        profile_info_file = os.path.join(fileDir, file)
        df_General  = pd.read_excel(profile_info_file, sheet_name="General")
        
        # If noData==TRUE in df_Light, skip file
        df_Light = pd.read_excel(profile_info_file, sheet_name="LIGHT")
        if df_Light['noData'].all():
            continue
        
        # Create table with 3 colunns: SEALTAG_NC_FILE, TAG_ID, PROFILE_ID
        tag_id = df_General['PlatformID']
        profile_num = df_General['Profile']
        file_nc = profile_info_file.replace('_ProfileInfo.xlsx', '_PROCESSED.nc')
        file_nc = [file_nc] * len(tag_id)
        lat = df_General['Lat']
        lon = df_General['Lon']
        date = df_General['Date']
        isDaytime = df_General['Daytime']
        
         # Append each profile to dataframe
        for i in range(len(tag_id)):
            new_row = pd.DataFrame({
                "SEALTAG_NC_FILE": [file_nc[i]],
                "TAG_ID": [tag_id.values[i]],
                "PROFILE_NUM": [profile_num.values[i]],
                "DATE": [date.values[i]],
                "LATITUDE": [lat.values[i]],
                "LONGITUDE": [lon.values[i]],
                "ISDAYTIME": [isDaytime.values[i]],
                "PROCESSED": [False]
            })
            df_profiles = pd.concat([df_profiles, new_row], ignore_index=True)
            
# If satellite_matchups.csv exists, find the latest processed profile and mark as PROCESSED
sat_matchups_file = "/Users/jweis/Library/CloudStorage/OneDrive-UniversityofTasmania/Work/Code/Library/SOCA_LIGHT_SEALS/SAT_MATCHUPS/ct182_satellite_matchups.csv"
if os.path.exists(sat_matchups_file):
    df_sat_matchups = pd.read_csv(sat_matchups_file, delimiter=",", skiprows=0)
    # Get latest processed profile and tag ID
    latest_processed_profilenum = df_sat_matchups["PROFILE_NUM"][len(df_sat_matchups)-1]
    latest_processed_tagID = df_sat_matchups["TAG_ID"][len(df_sat_matchups)-1]
    
    # Find row index in df_profiles with matching TAG_ID and PROFILE_NUM
    rowidx = df_profiles[
        (df_profiles["TAG_ID"] == latest_processed_tagID) &
        (df_profiles["PROFILE_NUM"] == latest_processed_profilenum)
    ].index[0]
    
    # Set PROCESSED=True for all rows prior to and including this row
    df_profiles.loc[0:rowidx, ["PROCESSED"]] = True

# Save dataframe to csv
profile_csv_file = "/Users/jweis/Library/CloudStorage/OneDrive-UniversityofTasmania/Work/Code/Library/SOCA_LIGHT_SEALS/SAT_MATCHUPS/ct182_profiles_to_match.csv"
df_profiles["DATE"] = pd.to_datetime(df_profiles["DATE"])
df_profiles.to_csv(profile_csv_file, index=False)

NameError: name 'pd' is not defined

### Test

In [ ]:
# TESTING CODE FOR SATELLITE MATCHUPS
profile_csv_file = "/Users/jweis/Library/CloudStorage/OneDrive-UniversityofTasmania/Work/Code/Library/SOCA_LIGHT_SEALS/SAT_MATCHUPS/ct182_profiles_to_match.csv"
all_df = pd.read_csv(
    profile_csv_file,
    delimiter=",",
    skiprows=0
)
all_df["DATE"] = pd.to_datetime(all_df["DATE"])
all_ids = all_df["TAG_ID"].values
all_profilnum = all_df["PROFILE_NUM"].values
all_lat = all_df["LATITUDE"].values
all_lon = all_df["LONGITUDE"].values
all_dates = all_df["DATE"].dt.strftime("%Y-%m-%d").tolist()
all_ncfiles = all_df["SEALTAG_NC_FILE"].values
all_isdaytime = all_df["ISDAYTIME"].values
all_processed = all_df["PROCESSED"].values

i = 5
date = all_dates[i]
lat = all_lat[i]
lon = all_lon[i]
df_Rrs = get_sat_matchups(
        start_date=date,
        end_date=date,
        latitude=lat,
        longitude=lon,
        wavelengths=[412, 443, 490, 555, 670],
        par="Rrs",
        sat="PACE_AOP",
        selected_dates=[date],
        )

print(df_Rrs)

### Get matchups

In [ ]:
from colorama import Fore, Back, Style

# Read in profile CSV file to get matchup information
profile_csv_file = "/Users/jweis/Library/CloudStorage/OneDrive-UniversityofTasmania/Work/Code/Library/SOCA_LIGHT_SEALS/SAT_MATCHUPS/ct182_profiles_to_match.csv"
all_df = pd.read_csv(
    profile_csv_file,
    delimiter=",",
    skiprows=0
)
all_df["DATE"] = pd.to_datetime(all_df["DATE"])
all_ids = all_df["TAG_ID"].values
all_profilnum = all_df["PROFILE_NUM"].values
all_lat = all_df["LATITUDE"].values
all_lon = all_df["LONGITUDE"].values
all_dates = all_df["DATE"].dt.strftime("%Y-%m-%d").tolist()
all_ncfiles = all_df["SEALTAG_NC_FILE"].values
all_isdaytime = all_df["ISDAYTIME"].values
all_processed = all_df["PROCESSED"].values

# Read in existing matchup CSV if it exists, otherwise create new one
matchup_csv_file = "/Users/jweis/Library/CloudStorage/OneDrive-UniversityofTasmania/Work/Code/Library/SOCA_LIGHT_SEALS/SAT_MATCHUPS/ct182_satellite_matchups.csv"
if not os.path.exists(matchup_csv_file):
    # Create empty dataframe with columns
    df_matchups = pd.DataFrame(columns=[
        "SEALTAG_NC_FILE",
        "TAG_ID",
        "PROFILE_NUM",
        "RRS412",
        "RRS443",
        "RRS490",
        "RRS555",
        "RRS670",
        "PAR",
        "KD490"
    ])
else:
    df_matchups = pd.read_csv(
        matchup_csv_file,
        delimiter=",",
        skiprows=0
    )

# Loop through each tag position and date
# go_to_next_day = False

for i, (lat, lon, date, tag, profile, ncfile, isdaytime, processed) in enumerate(zip(all_lat, all_lon, all_dates, all_ids, all_profilnum, all_ncfiles, all_isdaytime, all_processed)):
    
    # # Stop if we have at least 20 matchups (for testing)
    # if len(df_matchups) >= 20:
    #     break
    
    # Skip profiles that have already been processed
    if processed == True:
        print(f"{Fore.GREEN + Style.BRIGHT}Skipping tag {tag}, profile {profile}/{len(all_lat)}: lat={lat}, lon={lon}, date={date} (already processed){Style.RESET_ALL}")
        continue
    
    # Skip nighttime profiles
    if isdaytime == False:
        print(f"{Fore.RED + Style.BRIGHT}Skipping tag {tag}, profile {profile}/{len(all_lat)}: lat={lat}, lon={lon}, date={date} (nighttime profile){Style.RESET_ALL}")
        # Mark profile as processed in profile CSV
        df_profiles.at[i, "PROCESSED"] = True
        df_profiles.to_csv(profile_csv_file, index=False)
        continue
    
    print(f"{Fore.BLUE + Style.BRIGHT}Processing tag {tag}, profile {profile} ({i}/{len(all_lat)}): lat={lat}, lon={lon}, date={date}{Style.RESET_ALL}")

    # if go_to_next_day:
    #     if date == date_to_skip:
    #         print(f"Skipping date {date} as previously determined.")
    #         continue
    #     else:
    #         go_to_next_day = False
    
    # Extract Rrs from PACE AOP
    print(f"--> Extracting Rrs from PACE AOP")
    df_Rrs = get_sat_matchups(
        start_date=date,
        end_date=date,
        latitude=lat,
        longitude=lon,
        wavelengths=[412, 443, 490, 555, 670],
        par="Rrs",
        sat="PACE_AOP",
        selected_dates=[date],
        )
    
    # If valid Rrs data found, extract Kd from PACE IOP
    if len(df_Rrs)>0 and df_Rrs["sat_pixel_valid"].sum()>0:
        print(f"--> Extracting Kd from PACE IOP")
        df_Kd = get_sat_matchups(
            start_date=date,
            end_date=date,
            latitude=lat,
            longitude=lon,
            wavelengths=[490],
            par="Kd",
            sat="PACE_IOP",
            selected_dates=[date],
            )
        
        print(f"--> Extracting PAR from PACE PAR")
        df_PAR = get_sat_matchups(
            start_date=date,
            end_date=date,
            latitude=lat,
            longitude=lon,
            par="PAR",
            sat="PACE_PAR",
            selected_dates=[date],
            )
        
        # Merge Rrs, Kd, and PAR dataframes on sat_datetime
        df_satellite = pd.merge(
            df_Rrs,
            df_PAR,
            on=["sat_datetime", "sat_latitude", "sat_longitude", "sat_pixel_valid"],
            how="outer"
        )
        df_satellite = pd.merge(
            df_satellite,
            df_Kd,
            on=["sat_datetime", "sat_latitude", "sat_longitude", "sat_pixel_valid"],
            how="outer"
        )
        
        new_row = pd.DataFrame({
                "SEALTAG_NC_FILE": [ncfile],
                "TAG_ID": [tag],
                "PROFILE_NUM": [profile],
                "RRS412": [np.nanmean(df_satellite['sat_rrs413'].values)],
                "RRS443": [np.nanmean(df_satellite['sat_rrs442'].values)],
                "RRS490": [np.nanmean(df_satellite['sat_rrs490'].values)],
                "RRS555": [np.nanmean(df_satellite['sat_rrs555'].values)],
                "RRS670": [np.nanmean(df_satellite['sat_rrs670'].values)],
                "PAR": np.nanmean(df_satellite['sat_par'].values),
                "KD490": np.nanmean(df_satellite['sat_kd490'].values)
                
            })
        df_matchups = pd.concat([df_matchups, new_row], ignore_index=True)
        
        # Save intermediate results to CSV
        df_matchups.to_csv(matchup_csv_file, index=False)
        
        print(f"{Fore.GREEN}    Successfully extracted satellite data for date {date}.{Style.RESET_ALL}")
    else:
        print(f"{Fore.RED}    No valid satellite data for date {date}.{Style.RESET_ALL}")
    
    # Mark profile as processed in profile CSV
    df_profiles.at[i, "PROCESSED"] = True
    df_profiles.to_csv(profile_csv_file, index=False)
        
    # # If no valid data, skip to next day
    # if df_Rrs["sat_pixel_valid"].sum()==0:
    #     print(f"No valid satellite data for date {date}.")
    #     date_to_skip = date
    #     go_to_next_day = True


## Pre-2023 tags > VIIRS/MODIS matchups

### Create profile list

In [13]:
fileDir = "/Volumes/Data/Work/SEALTAGS/Prydz/FLUO_LIGHT/PROCESSED/NC"

df_profiles = pd.DataFrame(columns=["SEALTAG_NC_FILE", "TAG_ID", "PROFILE_NUM", "DATE", "LATITUDE", "LONGITUDE","ISDAYTIME","PROCESSED"])

# Loop through profile information excel files in directory
for file in sorted(os.listdir(fileDir)):
    if file.endswith(".xlsx") and not file.startswith("~$"):
        profile_info_file = os.path.join(fileDir, file)
        df_General  = pd.read_excel(profile_info_file, sheet_name="General")
        
        # If noData==TRUE in df_Light, skip file
        df_Light = pd.read_excel(profile_info_file, sheet_name="LIGHT")
        if df_Light['noData'].all():
            continue
        
        # Create table with 3 colunns: SEALTAG_NC_FILE, TAG_ID, PROFILE_ID
        tag_id = df_General['PlatformID']
        profile_num = df_General['Profile']
        file_nc = profile_info_file.replace('_ProfileInfo.xlsx', '_PROCESSED.nc')
        file_nc = [file_nc] * len(tag_id)
        lat = df_General['Lat']
        lon = df_General['Lon']
        date = df_General['Date']
        isDaytime = df_General['Daytime']
        
         # Append each profile to dataframe
        for i in range(len(tag_id)):
            new_row = pd.DataFrame({
                "SEALTAG_NC_FILE": [file_nc[i]],
                "TAG_ID": [tag_id.values[i]],
                "PROFILE_NUM": [profile_num.values[i]],
                "DATE": [date.values[i]],
                "LATITUDE": [lat.values[i]],
                "LONGITUDE": [lon.values[i]],
                "ISDAYTIME": [isDaytime.values[i]],
                "PROCESSED": [False]
            })
            df_profiles = pd.concat([df_profiles, new_row], ignore_index=True)
            
# If satellite_matchups.csv exists, find the latest processed profile and mark as PROCESSED
sat_matchups_file = "/Users/jweis/Library/CloudStorage/OneDrive-UniversityofTasmania/Work/Code/Library/SOCA_LIGHT_SEALS/SAT_MATCHUPS/2018_2023_satellite_matchups.csv"
if os.path.exists(sat_matchups_file):
    df_sat_matchups = pd.read_csv(sat_matchups_file, delimiter=",", skiprows=0)
    # Get latest processed profile and tag ID
    latest_processed_profilenum = df_sat_matchups["PROFILE_NUM"][len(df_sat_matchups)-1]
    latest_processed_tagID = df_sat_matchups["TAG_ID"][len(df_sat_matchups)-1]
    
    # Find row index in df_profiles with matching TAG_ID and PROFILE_NUM
    rowidx = df_profiles[
        (df_profiles["TAG_ID"] == latest_processed_tagID) &
        (df_profiles["PROFILE_NUM"] == latest_processed_profilenum)
    ].index[0]
    
    # Set PROCESSED=True for all rows prior to and including this row
    df_profiles.loc[0:rowidx, ["PROCESSED"]] = True

# Save dataframe to csv
profile_csv_file = "/Users/jweis/Library/CloudStorage/OneDrive-UniversityofTasmania/Work/Code/Library/SOCA_LIGHT_SEALS/SAT_MATCHUPS/2018_2023_profiles_to_match.csv"
df_profiles["DATE"] = pd.to_datetime(df_profiles["DATE"],format="%d/%m/%Y %H:%M")
df_profiles.to_csv(profile_csv_file, index=False)

/var/folders/t1/9ygzcs0j2ms2w3wgpmhfdhn1857szx/T/ipykernel_15681/2204866925.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_profiles = pd.concat([df_profiles, new_row], ignore_index=True)


### Test

In [37]:
# TESTING CODE FOR SATELLITE MATCHUPS
# Read in profile CSV file to get matchup information
cwd = os.getcwd()
profile_csv_file = cwd + "/2018_2023_profiles_to_match.csv"
df_profiles = pd.read_csv(
    profile_csv_file,
    delimiter=",",
    skiprows=0
)
df_profiles["DATE"] = pd.to_datetime(df_profiles["DATE"])
all_ids = df_profiles["TAG_ID"].values
all_profilnum = df_profiles["PROFILE_NUM"].values
all_lat = df_profiles["LATITUDE"].values
all_lon = df_profiles["LONGITUDE"].values
all_dates = df_profiles["DATE"].dt.strftime("%Y-%m-%d").tolist()
all_ncfiles = df_profiles["SEALTAG_NC_FILE"].values
all_isdaytime = df_profiles["ISDAYTIME"].values
all_processed = df_profiles["PROCESSED"].values

i = 1
date = all_dates[i]
lat = all_lat[i]
lon = all_lon[i]
df_Rrs = get_sat_matchups(
        start_date=date,
        end_date=date,
        latitude=lat,
        longitude=lon,
        wavelengths=[412, 443, 490, 555, 670],
        sat="AQUA",
        selected_dates=[date],
        )

print(df_Rrs)

    Filtered to 1 Granules.
    Running Granule: 2020-01-12 09:45:01+00:00
               sat_datetime  sat_latitude  sat_longitude  sat_pixel_valid  \
0 2020-01-12 09:45:01+00:00    -48.493919      69.103462               10   

   sat_rrs_band1  sat_rrs_band2  sat_rrs_band3  sat_rrs_band4  sat_rrs_band5  \
0       0.003841       0.004138       0.004736       0.001932       0.000274   

   sat_kd490    sat_par  
0    0.05848  66.298599  


### Get matchups

In [ ]:
from colorama import Fore, Back, Style

# Read in profile CSV file to get matchup information
cwd = os.getcwd()
profile_csv_file = os.path.join(cwd, "2018_2023_profiles_to_match.csv")
df_profiles = pd.read_csv(
    profile_csv_file,
    delimiter=",",
    skiprows=0
)
df_profiles["DATE"] = pd.to_datetime(df_profiles["DATE"])
all_ids = df_profiles["TAG_ID"].values
all_profilnum = df_profiles["PROFILE_NUM"].values
all_lat = df_profiles["LATITUDE"].values
all_lon = df_profiles["LONGITUDE"].values
all_dates = df_profiles["DATE"].dt.strftime("%Y-%m-%d").tolist()
all_ncfiles = df_profiles["SEALTAG_NC_FILE"].values
all_isdaytime = df_profiles["ISDAYTIME"].values
all_processed = df_profiles["PROCESSED"].values

# Read in existing matchup CSV if it exists, otherwise create new one
matchup_csv_file = os.path.join(cwd, "2018_2023_satellite_matchups.csv")
if not os.path.exists(matchup_csv_file):
    # Create empty dataframe with columns
    df_matchups = pd.DataFrame(columns=[
        "SEALTAG_NC_FILE",
        "TAG_ID",
        "PROFILE_NUM",
        "RRS412",
        "RRS443",
        "RRS490",
        "RRS555",
        "RRS670",
        "KD490",
        "PAR",
        "SATELLITE"
    ])
else:
    df_matchups = pd.read_csv(
        matchup_csv_file,
        delimiter=",",
        skiprows=0
    )

# Loop through each tag position and date
for i, (lat, lon, date, tag, profile, ncfile, isdaytime, processed) in enumerate(zip(all_lat, all_lon, all_dates, all_ids, all_profilnum, all_ncfiles, all_isdaytime, all_processed)):
    # # Stop if we have at least 20 matchups (for testing)
    # if len(df_matchups) >= 1:
    #     break
    
    # Skip profiles that have already been processed
    if processed == True:
        print(f"{Fore.GREEN + Style.BRIGHT}Skipping tag {tag}, profile {profile}/{len(all_lat)}: lat={lat}, lon={lon}, date={date} (already processed){Style.RESET_ALL}")
        continue
    
    # Skip nighttime profiles
    if isdaytime == False:
        print(f"{Fore.RED + Style.BRIGHT}Skipping tag {tag}, profile {profile}/{len(all_lat)}: lat={lat}, lon={lon}, date={date} (nighttime profile){Style.RESET_ALL}")
        # Mark profile as processed in profile CSV
        df_profiles.at[i, "PROCESSED"] = True
        df_profiles.to_csv(profile_csv_file, index=False)
        continue
    
    print(f"{Fore.BLUE + Style.BRIGHT}Processing tag {tag}, profile {profile} ({i}/{len(all_lat)}): lat={lat}, lon={lon}, date={date}{Style.RESET_ALL}")
    
    # 1) VIIRS SUOMI-NPP
    print(f"--> Extracting Rrs from SUOMI-NPP")
    df_data = get_sat_matchups(
        start_date=date,
        end_date=date,
        latitude=lat,
        longitude=lon,
        wavelengths=[412, 443, 490, 555, 670],
        sat="SUOMI-NPP",
        selected_dates=[date],
        )
    # If valid Rrs data found, extract Kd from PACE IOP
    if len(df_data)>0 and df_data["sat_pixel_valid"].sum()>0 and not (df_data["sat_rrs_band1"].isna().all() or df_data["sat_rrs_band2"].isna().all() or df_data["sat_rrs_band3"].isna().all() or df_data["sat_rrs_band4"].isna().all() or df_data["sat_rrs_band5"].isna().all() or df_data["sat_kd490"].isna().all() or df_data["sat_par"].isna().all()):
        new_row = pd.DataFrame({
                "SEALTAG_NC_FILE": [ncfile],
                "TAG_ID": [tag],
                "PROFILE_NUM": [profile],
                "RRS412": [np.nanmean(df_data['sat_rrs_band1'].values)],
                "RRS443": [np.nanmean(df_data['sat_rrs_band2'].values)],
                "RRS490": [np.nanmean(df_data['sat_rrs_band3'].values)],
                "RRS555": [np.nanmean(df_data['sat_rrs_band4'].values)],
                "RRS670": [np.nanmean(df_data['sat_rrs_band5'].values)],
                "KD490": np.nanmean(df_data['sat_kd490'].values),
                "PAR": np.nanmean(df_data['sat_par'].values),
                "SATELLITE": "VIIRS_SUOMI-NPP"
            })
        df_matchups = pd.concat([df_matchups, new_row], ignore_index=True)
        
        # Save intermediate results to CSV
        df_matchups.to_csv(matchup_csv_file, index=False)
        
        print(f"{Fore.GREEN}    Successfully extracted VIIRS SUOMI-NPP data.{Style.RESET_ALL}")
    else:
        print(f"{Fore.RED}    No valid VIIRS SUOMI-NPP data.{Style.RESET_ALL}")
        
    # 2) VIIRS NOAA-20
    print(f"--> Extracting Rrs from VIIRS NOAA-20")
    df_data = get_sat_matchups(
        start_date=date,
        end_date=date,
        latitude=lat,
        longitude=lon,
        wavelengths=[412, 443, 490, 555, 670],
        sat="NOAA-20",
        selected_dates=[date],
        )
    # If valid Rrs data found, extract Kd from PACE IOP
    if len(df_data)>0 and df_data["sat_pixel_valid"].sum()>0 and not (df_data["sat_rrs_band1"].isna().all() or df_data["sat_rrs_band2"].isna().all() or df_data["sat_rrs_band3"].isna().all() or df_data["sat_rrs_band4"].isna().all() or df_data["sat_rrs_band5"].isna().all() or df_data["sat_kd490"].isna().all() or df_data["sat_par"].isna().all()):
        new_row = pd.DataFrame({
                "SEALTAG_NC_FILE": [ncfile],
                "TAG_ID": [tag],
                "PROFILE_NUM": [profile],
                "RRS412": [np.nanmean(df_data['sat_rrs_band1'].values)],
                "RRS443": [np.nanmean(df_data['sat_rrs_band2'].values)],
                "RRS490": [np.nanmean(df_data['sat_rrs_band3'].values)],
                "RRS555": [np.nanmean(df_data['sat_rrs_band4'].values)],
                "RRS670": [np.nanmean(df_data['sat_rrs_band5'].values)],
                "KD490": np.nanmean(df_data['sat_kd490'].values),
                "PAR": np.nanmean(df_data['sat_par'].values),
                "SATELLITE": "VIIRS_NOAA-20"
            })
        df_matchups = pd.concat([df_matchups, new_row], ignore_index=True)
        
        # Save intermediate results to CSV
        df_matchups.to_csv(matchup_csv_file, index=False)
        
        print(f"{Fore.GREEN}    Successfully extracted VIIRS NOAA-20 data.{Style.RESET_ALL}")
    else:
        print(f"{Fore.RED}    No valid VIIRS NOAA-20 data.{Style.RESET_ALL}")
        
    # 3) VIIRS NOAA-21
    print(f"--> Extracting Rrs from VIIRS NOAA-21")
    df_data = get_sat_matchups(
        start_date=date,
        end_date=date,
        latitude=lat,
        longitude=lon,
        wavelengths=[412, 443, 490, 555, 670],
        sat="NOAA-21",
        selected_dates=[date],
        )
    # If valid Rrs data found, extract Kd from PACE IOP
    if len(df_data)>0 and df_data["sat_pixel_valid"].sum()>0 and not (df_data["sat_rrs_band1"].isna().all() or df_data["sat_rrs_band2"].isna().all() or df_data["sat_rrs_band3"].isna().all() or df_data["sat_rrs_band4"].isna().all() or df_data["sat_rrs_band5"].isna().all() or df_data["sat_kd490"].isna().all() or df_data["sat_par"].isna().all()):
        new_row = pd.DataFrame({
                "SEALTAG_NC_FILE": [ncfile],
                "TAG_ID": [tag],
                "PROFILE_NUM": [profile],
                "RRS412": [np.nanmean(df_data['sat_rrs_band1'].values)],
                "RRS443": [np.nanmean(df_data['sat_rrs_band2'].values)],
                "RRS490": [np.nanmean(df_data['sat_rrs_band3'].values)],
                "RRS555": [np.nanmean(df_data['sat_rrs_band4'].values)],
                "RRS670": [np.nanmean(df_data['sat_rrs_band5'].values)],
                "KD490": np.nanmean(df_data['sat_kd490'].values),
                "PAR": np.nanmean(df_data['sat_par'].values),
                "SATELLITE": "VIIRS_NOAA-21"
            })
        df_matchups = pd.concat([df_matchups, new_row], ignore_index=True)
        
        # Save intermediate results to CSV
        df_matchups.to_csv(matchup_csv_file, index=False)
        
        print(f"{Fore.GREEN}    Successfully extracted VIIRS NOAA-21 data.{Style.RESET_ALL}")
    else:
        print(f"{Fore.RED}    No valid VIIRS NOAA-21 data.{Style.RESET_ALL}")
        
    # 4) MODIS AQUA
    print(f"--> Extracting Rrs from MODIS AQUA")
    df_data = get_sat_matchups(
        start_date=date,
        end_date=date,
        latitude=lat,
        longitude=lon,
        wavelengths=[412, 443, 490, 555, 670],
        sat="AQUA",
        selected_dates=[date],
        )
    # If valid Rrs data found, extract Kd from PACE IOP
    if len(df_data)>0 and df_data["sat_pixel_valid"].sum()>0 and not (df_data["sat_rrs_band1"].isna().all() or df_data["sat_rrs_band2"].isna().all() or df_data["sat_rrs_band3"].isna().all() or df_data["sat_rrs_band4"].isna().all() or df_data["sat_rrs_band5"].isna().all() or df_data["sat_kd490"].isna().all() or df_data["sat_par"].isna().all()):
        new_row = pd.DataFrame({
                "SEALTAG_NC_FILE": [ncfile],
                "TAG_ID": [tag],
                "PROFILE_NUM": [profile],
                "RRS412": [np.nanmean(df_data['sat_rrs_band1'].values)],
                "RRS443": [np.nanmean(df_data['sat_rrs_band2'].values)],
                "RRS490": [np.nanmean(df_data['sat_rrs_band3'].values)],
                "RRS555": [np.nanmean(df_data['sat_rrs_band4'].values)],
                "RRS670": [np.nanmean(df_data['sat_rrs_band5'].values)],
                "KD490": np.nanmean(df_data['sat_kd490'].values),
                "PAR": np.nanmean(df_data['sat_par'].values),
                "SATELLITE": "MODIS_AQUA"
            })
        df_matchups = pd.concat([df_matchups, new_row], ignore_index=True)
        
        # Save intermediate results to CSV
        df_matchups.to_csv(matchup_csv_file, index=False)
        
        print(f"{Fore.GREEN}    Successfully extracted MODIS AQUA data.{Style.RESET_ALL}")
    else:
        print(f"{Fore.RED}    No valid MODIS AQUA data.{Style.RESET_ALL}")
    
    # Mark profile as processed in profile CSV
    df_profiles.at[i, "PROCESSED"] = True
    df_profiles.to_csv(profile_csv_file, index=False)

### Parallel Option 1

In [ ]:
from colorama import Fore, Back, Style
import concurrent.futures
from functools import partial

def extract_satellite_data(sat_name, date, lat, lon, wavelengths=[412, 443, 490, 555, 670]):
    """Extract data from a single satellite for given coordinates and date."""
    print(f"    --> Extracting Rrs from {sat_name}")
    
    try:
        df_data = get_sat_matchups(
            start_date=date,
            end_date=date,
            latitude=lat,
            longitude=lon,
            wavelengths=wavelengths,
            sat=sat_name,
            selected_dates=[date],
        )
        
        # Check if valid data exists
        if (len(df_data) > 0 and df_data["sat_pixel_valid"].sum() > 0 and 
            not (df_data["sat_rrs_band1"].isna().all() or df_data["sat_rrs_band2"].isna().all() or 
                 df_data["sat_rrs_band3"].isna().all() or df_data["sat_rrs_band4"].isna().all() or 
                 df_data["sat_rrs_band5"].isna().all() or df_data["sat_kd490"].isna().all() or 
                 df_data["sat_par"].isna().all())):
            
            result = {
                "RRS412": np.nanmean(df_data['sat_rrs_band1'].values),
                "RRS443": np.nanmean(df_data['sat_rrs_band2'].values),
                "RRS490": np.nanmean(df_data['sat_rrs_band3'].values),
                "RRS555": np.nanmean(df_data['sat_rrs_band4'].values),
                "RRS670": np.nanmean(df_data['sat_rrs_band5'].values),
                "KD490": np.nanmean(df_data['sat_kd490'].values),
                "PAR": np.nanmean(df_data['sat_par'].values),
                "SATELLITE": sat_name.replace("-", "_"),
                "success": True
            }
            print(f"{Fore.GREEN}    Successfully extracted {sat_name} data.{Style.RESET_ALL}")
            return result
        else:
            print(f"{Fore.RED}    No valid {sat_name} data.{Style.RESET_ALL}")
            return {"success": False, "satellite": sat_name}
            
    except Exception as e:
        print(f"{Fore.RED}    Error extracting {sat_name} data: {e}{Style.RESET_ALL}")
        return {"success": False, "satellite": sat_name, "error": str(e)}

# Read in profile CSV file to get matchup information
cwd = os.getcwd()
profile_csv_file = os.path.join(cwd, "2018_2023_profiles_to_match.csv")
df_profiles = pd.read_csv(
    profile_csv_file,
    delimiter=",",
    skiprows=0
)
df_profiles["DATE"] = pd.to_datetime(df_profiles["DATE"])
all_ids = df_profiles["TAG_ID"].values
all_profilnum = df_profiles["PROFILE_NUM"].values
all_lat = df_profiles["LATITUDE"].values
all_lon = df_profiles["LONGITUDE"].values
all_dates = df_profiles["DATE"].dt.strftime("%Y-%m-%d").tolist()
all_ncfiles = df_profiles["SEALTAG_NC_FILE"].values
all_isdaytime = df_profiles["ISDAYTIME"].values
all_processed = df_profiles["PROCESSED"].values

# Read in existing matchup CSV if it exists, otherwise create new one
cwd = os.getcwd()
matchup_csv_file = os.path.join(cwd, "2018_2023_satellite_matchups.csv")
if not os.path.exists(matchup_csv_file):
    # Create empty dataframe with columns
    df_matchups = pd.DataFrame(columns=[
        "SEALTAG_NC_FILE",
        "TAG_ID",
        "PROFILE_NUM",
        "RRS412",
        "RRS443",
        "RRS490",
        "RRS555",
        "RRS670",
        "KD490",
        "PAR",
        "SATELLITE"
    ])
else:
    df_matchups = pd.read_csv(
        matchup_csv_file,
        delimiter=",",
        skiprows=0
    )

# List of satellites to process
satellites = ["NOAA-20", "SUOMI-NPP", "NOAA-21", "AQUA"]

# Loop through each tag position and date
for i, (lat, lon, date, tag, profile, ncfile, isdaytime, processed) in enumerate(zip(all_lat, all_lon, all_dates, all_ids, all_profilnum, all_ncfiles, all_isdaytime, all_processed)):
    
    # Skip profiles that have already been processed
    if processed == True:
        print(f"{Fore.GREEN + Style.BRIGHT}Skipping tag {tag}, profile {profile} ({i}/{len(all_lat)}): lat={lat}, lon={lon}, date={date} (already processed){Style.RESET_ALL}")
        continue
    
    # Skip nighttime profiles
    if isdaytime == False:
        print(f"{Fore.RED + Style.BRIGHT}Skipping tag {tag}, profile {profile} ({i}/{len(all_lat)}): lat={lat}, lon={lon}, date={date} (nighttime profile){Style.RESET_ALL}")
        # Mark profile as processed in profile CSV
        df_profiles.at[i, "PROCESSED"] = True
        df_profiles.to_csv(profile_csv_file, index=False)
        continue
    
    print(f"{Fore.BLUE + Style.BRIGHT}Processing tag {tag}, profile {profile} ({i}/{len(all_lat)}): lat={lat}, lon={lon}, date={date}{Style.RESET_ALL}")
    
    # Process all satellites in parallel
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
        # Create partial function with fixed parameters
        extract_func = partial(extract_satellite_data, date=date, lat=lat, lon=lon)
        
        # Submit all satellite extraction tasks
        future_to_satellite = {executor.submit(extract_func, sat): sat for sat in satellites}
        
        # Collect results as they complete
        for future in concurrent.futures.as_completed(future_to_satellite):
            result = future.result()
            
            # If successful, add to dataframe
            if result.get("success", False):
                new_row = pd.DataFrame({
                    "SEALTAG_NC_FILE": [ncfile],
                    "TAG_ID": [tag],
                    "PROFILE_NUM": [profile],
                    "RRS412": [result["RRS412"]],
                    "RRS443": [result["RRS443"]],
                    "RRS490": [result["RRS490"]],
                    "RRS555": [result["RRS555"]],
                    "RRS670": [result["RRS670"]],
                    "KD490": [result["KD490"]],
                    "PAR": [result["PAR"]],
                    "SATELLITE": [result["SATELLITE"]]
                })
                df_matchups = pd.concat([df_matchups, new_row], ignore_index=True)
    
    # Save intermediate results to CSV after each profile
    df_matchups.to_csv(matchup_csv_file, index=False)
    
    # Mark profile as processed in profile CSV
    df_profiles.at[i, "PROCESSED"] = True
    df_profiles.to_csv(profile_csv_file, index=False)

Skipping tag ct157-F116-20, profile 1 (0/4343): lat=-48.7066, lon=69.83, date=2020-01-12 (already processed)
Skipping tag ct157-F116-20, profile 2 (1/4343): lat=-48.4935, lon=69.105, date=2020-01-12 (already processed)
Skipping tag ct157-F116-20, profile 3 (2/4343): lat=-48.3914, lon=68.1434, date=2020-01-13 (already processed)
Skipping tag ct157-F116-20, profile 4 (3/4343): lat=-48.3219, lon=67.6565, date=2020-01-13 (already processed)
Skipping tag ct157-F116-20, profile 5 (4/4343): lat=-48.4779, lon=66.9255, date=2020-01-14 (already processed)
Skipping tag ct157-F116-20, profile 6 (5/4343): lat=-48.2997, lon=66.5506, date=2020-01-14 (already processed)
Skipping tag ct157-F116-20, profile 7 (6/4343): lat=-48.0698, lon=65.5466, date=2020-01-15 (already processed)
Skipping tag ct157-F116-20, profile 8 (7/4343): lat=-48.1352, lon=65.409, date=2020-01-15 (already processed)
Skipping tag ct157-F116-20, profile 9 (8/4343): lat=-48.1141, lon=64.2633, date=2020-01-16 (already processed)
Skipp

### Parallel Option 2 - Process Profiles in Parallel

In [ ]:
from colorama import Fore, Back, Style
import concurrent.futures
import threading
import time

# Thread-safe lock for file operations
file_lock = threading.Lock()

def process_single_profile(profile_data, satellites, matchup_csv_file, profile_csv_file, df_profiles, wavelengths=[412, 443, 490, 555, 670]):
    """Process a single profile through all satellites sequentially."""
    i, lat, lon, date, tag, profile, ncfile, isdaytime, processed = profile_data
    
    # Skip profiles that have already been processed
    if processed == True:
        print(f"{Fore.GREEN + Style.BRIGHT}[Thread] Skipping tag {tag}, profile {profile} ({i}): already processed{Style.RESET_ALL}")
        return []
    
    # Skip nighttime profiles
    if isdaytime == False:
        print(f"{Fore.RED + Style.BRIGHT}[Thread] Skipping tag {tag}, profile {profile} ({i}): nighttime profile{Style.RESET_ALL}")
        # Mark profile as processed in profile CSV (thread-safe)
        with file_lock:
            df_profiles.at[i, "PROCESSED"] = True
            df_profiles.to_csv(profile_csv_file, index=False)
        return []
    
    print(f"{Fore.BLUE + Style.BRIGHT}[Thread] Processing tag {tag}, profile {profile} ({i}): lat={lat}, lon={lon}, date={date}{Style.RESET_ALL}")
    
    # Store results for this profile
    profile_results = []
    
    # Process all satellites sequentially for this profile
    for sat_name in satellites:
        print(f"    [Thread] --> Extracting Rrs from {sat_name}")
        
        try:
            df_data = get_sat_matchups(
                start_date=date,
                end_date=date,
                latitude=lat,
                longitude=lon,
                wavelengths=wavelengths,
                sat=sat_name,
                selected_dates=[date],
            )
            
            # Check if valid data exists
            if (len(df_data) > 0 and df_data["sat_pixel_valid"].sum() > 0 and 
                not (df_data["sat_rrs_band1"].isna().all() or df_data["sat_rrs_band2"].isna().all() or 
                     df_data["sat_rrs_band3"].isna().all() or df_data["sat_rrs_band4"].isna().all() or 
                     df_data["sat_rrs_band5"].isna().all() or df_data["sat_kd490"].isna().all() or 
                     df_data["sat_par"].isna().all())):
                
                result = {
                    "SEALTAG_NC_FILE": ncfile,
                    "TAG_ID": tag,
                    "PROFILE_NUM": profile,
                    "RRS412": np.nanmean(df_data['sat_rrs_band1'].values),
                    "RRS443": np.nanmean(df_data['sat_rrs_band2'].values),
                    "RRS490": np.nanmean(df_data['sat_rrs_band3'].values),
                    "RRS555": np.nanmean(df_data['sat_rrs_band4'].values),
                    "RRS670": np.nanmean(df_data['sat_rrs_band5'].values),
                    "KD490": np.nanmean(df_data['sat_kd490'].values),
                    "PAR": np.nanmean(df_data['sat_par'].values),
                    "SATELLITE": sat_name.replace("-", "_")
                }
                
                profile_results.append(result)
                print(f"{Fore.GREEN}    [Thread] Successfully extracted {sat_name} data.{Style.RESET_ALL}")
            else:
                print(f"{Fore.RED}    [Thread] No valid {sat_name} data.{Style.RESET_ALL}")
                
        except Exception as e:
            print(f"{Fore.RED}    [Thread] Error extracting {sat_name} data: {e}{Style.RESET_ALL}")
    
    # Mark profile as processed (thread-safe)
    with file_lock:
        df_profiles.at[i, "PROCESSED"] = True
        df_profiles.to_csv(profile_csv_file, index=False)
        
        # Save results to CSV if any were found (thread-safe)
        if profile_results:
            df_new_results = pd.DataFrame(profile_results)
            
            # Read existing results and append new ones
            if os.path.exists(matchup_csv_file):
                df_existing = pd.read_csv(matchup_csv_file, delimiter=",", skiprows=0)
                df_combined = pd.concat([df_existing, df_new_results], ignore_index=True)
            else:
                df_combined = df_new_results
            
            df_combined.to_csv(matchup_csv_file, index=False)
            print(f"{Fore.CYAN}[Thread] Saved {len(profile_results)} results for profile {profile}{Style.RESET_ALL}")
    
    return profile_results


# Read in profile CSV file to get matchup information
cwd = os.getcwd()
profile_csv_file = os.path.join(cwd, "2018_2023_profiles_to_match.csv")
df_profiles = pd.read_csv(
    profile_csv_file,
    delimiter=",",
    skiprows=0
)
df_profiles["DATE"] = pd.to_datetime(df_profiles["DATE"])

# Create arrays of all profile data
all_ids = df_profiles["TAG_ID"].values
all_profilnum = df_profiles["PROFILE_NUM"].values
all_lat = df_profiles["LATITUDE"].values
all_lon = df_profiles["LONGITUDE"].values
all_dates = df_profiles["DATE"].dt.strftime("%Y-%m-%d").tolist()
all_ncfiles = df_profiles["SEALTAG_NC_FILE"].values
all_isdaytime = df_profiles["ISDAYTIME"].values
all_processed = df_profiles["PROCESSED"].values

# Set up output file
matchup_csv_file = os.path.join(cwd, "2018_2023_satellite_matchups.csv")
if not os.path.exists(matchup_csv_file):
    # Create empty dataframe with columns
    df_matchups = pd.DataFrame(columns=[
        "SEALTAG_NC_FILE",
        "TAG_ID",
        "PROFILE_NUM",
        "RRS412",
        "RRS443",
        "RRS490",
        "RRS555",
        "RRS670",
        "KD490",
        "PAR",
        "SATELLITE"
    ])
    df_matchups.to_csv(matchup_csv_file, index=False)

# List of satellites to process
satellites = ["NOAA-20", "SUOMI-NPP", "NOAA-21", "AQUA"]

# Create profile data tuples for processing
profile_data_list = []
for i, (lat, lon, date, tag, profile, ncfile, isdaytime, processed) in enumerate(
    zip(all_lat, all_lon, all_dates, all_ids, all_profilnum, all_ncfiles, all_isdaytime, all_processed)
):
    profile_data_list.append((i, lat, lon, date, tag, profile, ncfile, isdaytime, processed))

# Process profiles in parallel
print(f"{Fore.MAGENTA + Style.BRIGHT}Starting parallel processing of {len(profile_data_list)} profiles using {min(4, len(profile_data_list))} workers...{Style.RESET_ALL}")

start_time = time.time()
total_results = 0

# Use ProcessPoolExecutor for CPU-bound tasks, ThreadPoolExecutor for I/O-bound tasks
# Since satellite data extraction involves network I/O, we'll use ThreadPoolExecutor
with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
    # Submit all profile processing tasks
    future_to_profile = {
        executor.submit(
            process_single_profile, 
            profile_data, 
            satellites, 
            matchup_csv_file, 
            profile_csv_file, 
            df_profiles
        ): profile_data[4] for profile_data in profile_data_list  # profile_data[4] is the profile number
    }
    
    # Collect results as they complete
    for future in concurrent.futures.as_completed(future_to_profile):
        profile_num = future_to_profile[future]
        try:
            results = future.result()
            total_results += len(results)
            print(f"{Fore.YELLOW}Completed profile {profile_num}, found {len(results)} satellite matches{Style.RESET_ALL}")
        except Exception as e:
            print(f"{Fore.RED}Profile {profile_num} generated an exception: {e}{Style.RESET_ALL}")

end_time = time.time()
processing_time = end_time - start_time

print(f"{Fore.MAGENTA + Style.BRIGHT}")
print(f"Parallel processing completed!")
print(f"Total processing time: {processing_time:.2f} seconds")
print(f"Total satellite matches found: {total_results}")
print(f"Average time per profile: {processing_time/len(profile_data_list):.2f} seconds")
print(f"{Style.RESET_ALL}")

### Comparison of Parallel Processing Approaches

**Option 1 - Satellites in Parallel per Profile:**
- Processes profiles sequentially, but extracts data from all satellites simultaneously for each profile
- Better for: Limited profiles, many satellites, when network bandwidth is high
- Pros: Faster per-profile processing, simpler progress tracking
- Cons: May overwhelm satellite data services with concurrent requests

**Option 2 - Profiles in Parallel:**  
- Processes multiple profiles simultaneously, each extracting satellite data sequentially
- Better for: Many profiles, moderate satellite count, when you want to maximize throughput
- Pros: Better overall throughput, more resilient to individual failures, respects API rate limits per profile
- Cons: More complex error handling, potential resource contention

**Usage Recommendations:**
- Use Option 1 when processing < 100 profiles with good network connectivity
- Use Option 2 when processing > 100 profiles or when satellite APIs have rate limits
- Adjust `max_workers` parameter based on your system resources and API constraints